# Descriptive statistics figures (§3.4)

This notebook produces the descriptive-statistics figures for the thesis from
`hive_metastore.gold.gold_dataset` (unscaled values). It is organised as:

1. **Shared setup** — load the data once and compute load ratios. All figure cells below
   depend on this cell having run first.
2. **One cell per figure**, in thesis order (load ratios, single-transformer views,
   weather, overload rate, seasonal, events).
3. **Label balance** figures (train/test split, label-by-season) from
   `gold_features`, which has the labels.
4. **Spatial breakdown** by district and concelho.

## 1. Shared setup

Load the dataset once, filter zero readings, and compute the current and voltage load ratios. Run this before any figure cell below.

In [0]:
# =============================================================================
# Thesis §3.4 — All Descriptive Statistics Figures
# =============================================================================
# Source: hive_metastore.gold.gold_dataset (unscaled values)
# All figures filter out rows where current=0 or voltage=0
# Run each CELL block separately in Databricks
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

SRC_TABLE = "hive_metastore.gold.gold_dataset"
TS_COL = "DATE"

# ─── SHARED DATA LOAD ────────────────────────────────────────────────────────
# Run this cell first — all figures depend on it

df = spark.read.table(SRC_TABLE)
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))

df = df.withColumn(
    "load_ratio_c",
    F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
).withColumn(
    "load_ratio_v",
    F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
)

print(f"Rows after zero filter: {df.count():,}")



## 2. Fleet daily average load ratios

Daily mean current and voltage load ratio across all transformers, with a 7-day rolling mean and the overload threshold.

In [0]:
# =============================================================================
# CELL 1 — Figure 4: Fleet-Wide Daily Average Load Ratios (2-panel)
# =============================================================================

daily_avg = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.mean("load_ratio_c").alias("mean_lr_c"),
        F.mean("load_ratio_v").alias("mean_lr_v"),
    )
    .orderBy("date")
    .toPandas()
)
daily_avg["date"] = pd.to_datetime(daily_avg["date"])
daily_avg = daily_avg.set_index("date").sort_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# Panel A — Current
ax1 = axes[0]
ax1.plot(daily_avg.index, daily_avg["mean_lr_c"], linewidth=0.8, color="#1f77b4", alpha=0.7, label="Daily mean")
ax1.plot(daily_avg.index, daily_avg["mean_lr_c"].rolling(7, center=True).mean(), linewidth=2, color="#1f77b4", label="7-day rolling mean")
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")
ax1.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax1.set_ylim(0, min(daily_avg["mean_lr_c"].max() * 1.3, 1.2))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Current Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

# Panel B — Voltage
ax2 = axes[1]
ax2.plot(daily_avg.index, daily_avg["mean_lr_v"], linewidth=0.8, color="#ff7f0e", alpha=0.7, label="Daily mean")
ax2.plot(daily_avg.index, daily_avg["mean_lr_v"].rolling(7, center=True).mean(), linewidth=2, color="#ff7f0e", label="7-day rolling mean")
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (V / V_limit)", fontsize=11)
ax2.set_ylim(0, min(daily_avg["mean_lr_v"].max() * 1.3, 1.2))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Voltage Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig4_avg_load_ratios.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig4_avg_load_ratios.png")



## 3. Single transformer — full year

Load ratio over the full year for one representative transformer, with overload points marked.

In [0]:
# =============================================================================
# CELL 2 — Figure 5: Single Transformer Full Year (BSCVER3TP1-0)
# =============================================================================

TRANSFORMER_ID = "BSCVER3TP1-0"

df_single = (
    df.filter(F.col("ID_prefix") == TRANSFORMER_ID)
    .select(TS_COL, "load_ratio_c")
    .orderBy(TS_COL)
    .toPandas()
)
df_single[TS_COL] = pd.to_datetime(df_single[TS_COL])
df_single = df_single.set_index(TS_COL).sort_index()

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
ax.plot(df_single.index, df_single["load_ratio_c"], linewidth=0.3, color="#1f77b4", alpha=0.7, label="Load ratio (current)")
ax.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

overload_mask = df_single["load_ratio_c"] > 1
ax.scatter(df_single.index[overload_mask], df_single.loc[overload_mask, "load_ratio_c"],
           color="#d62728", s=3, alpha=0.6, zorder=5, label="Overload events")

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax.set_ylim(0, min(df_single["load_ratio_c"].max() * 1.1, 3.0))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f"Transformer {TRANSFORMER_ID} — Load Ratio Over 12 Months", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig5_full_year.png", dpi=300, bbox_inches="tight")
plt.show()

n_overloads = overload_mask.sum()
n_total = len(df_single)
print(f"Saved: /dbfs/tmp/thesis_fig5_full_year.png")
print(f"Overloads: {n_overloads:,} / {n_total:,} ({100*n_overloads/n_total:.2f}%)")



## 4. Single transformer — two-week detail

A two-week zoom on the same transformer at 15-minute resolution.

> Depends on the previous cell (it reuses `df_single`).

In [0]:
# =============================================================================
# CELL 3 — Figure 6: Single Transformer 2-Week Zoom (1–15 Jan 2024)
# =============================================================================

zoom_start = "2024-01-01"
zoom_end = "2024-01-15"
df_zoom = df_single.loc[zoom_start:zoom_end]

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
ax.plot(df_zoom.index, df_zoom["load_ratio_c"], linewidth=0.8, color="#1f77b4", alpha=0.9, label="Load ratio (current)")
ax.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

overload_mask_zoom = df_zoom["load_ratio_c"] > 1
if overload_mask_zoom.sum() > 0:
    ax.scatter(df_zoom.index[overload_mask_zoom], df_zoom.loc[overload_mask_zoom, "load_ratio_c"],
               color="#d62728", s=20, alpha=0.8, zorder=5, label="Overload events")
    for idx in df_zoom.index[overload_mask_zoom]:
        ax.axvspan(idx, idx + pd.Timedelta(minutes=15), color="#d62728", alpha=0.1)

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax.set_ylim(0, min(df_zoom["load_ratio_c"].max() * 1.1, 2.5))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_minor_locator(mdates.HourLocator(interval=12))
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f"Transformer {TRANSFORMER_ID} — 2-Week Detail (1–15 January 2024)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig6_zoom.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: /dbfs/tmp/thesis_fig6_zoom.png  |  Overloads in window: {overload_mask_zoom.sum()}")



## 5. Weather variables over the year

Daily mean of the four weather variables, each with a 7-day rolling mean.

In [0]:
# =============================================================================
# CELL 4 — Figure 7: Weather 4-Panel
# =============================================================================

WEATHER_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m_per_s"
]

daily_weather = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(*[F.mean(c).alias(c) for c in WEATHER_COLS])
    .orderBy("date")
    .toPandas()
)
daily_weather["date"] = pd.to_datetime(daily_weather["date"])
daily_weather = daily_weather.set_index("date").sort_index()

labels = ["(a) Mean Air Temperature (°C)", "(b) Relative Humidity (%)", "(c) Precipitation (mm)", "(d) Wind Speed (m/s)"]
colors = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), dpi=150, sharex=True)

for i, (col, label, color) in enumerate(zip(WEATHER_COLS, labels, colors)):
    ax = axes[i]
    ax.plot(daily_weather.index, daily_weather[col], linewidth=0.8, color=color, alpha=0.7)
    rolling = daily_weather[col].rolling(7, center=True).mean()
    ax.plot(daily_weather.index, rolling, linewidth=2, color=color)
    ax.set_ylabel(label.split(") ")[1], fontsize=10)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date", fontsize=11)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig7_weather.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig7_weather.png")



## 6. Daily overload rate with train/test split

Daily current and voltage overload rate (7-day rolling), with the 1 April 2024 split marked.

In [0]:
# =============================================================================
# CELL 5 — Figure 8: Daily Overload Rate (Current vs Voltage) + Train/Test Split
# =============================================================================

daily_overload = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("rate_c"),
        F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("rate_v"),
    )
    .orderBy("date")
    .toPandas()
)
daily_overload["date"] = pd.to_datetime(daily_overload["date"])
daily_overload = daily_overload.set_index("date").sort_index()
daily_overload["rate_c"] *= 100
daily_overload["rate_v"] *= 100

fig, ax = plt.subplots(figsize=(12, 5), dpi=150)

ax.plot(daily_overload.index, daily_overload["rate_c"].rolling(7, center=True).mean(),
        linewidth=2, color="#1f77b4", label="Current overloads")
ax.plot(daily_overload.index, daily_overload["rate_v"].rolling(7, center=True).mean(),
        linewidth=2, color="#ff7f0e", label="Voltage overloads")

# Train/test split line
split_date = pd.Timestamp("2024-04-01")
ax.axvline(x=split_date, color="black", linestyle="--", linewidth=1.5, label="Train/test split")
ax.text(split_date - pd.Timedelta(days=15), ax.get_ylim()[1] * 0.92, "Training", ha="right", fontsize=10, fontstyle="italic")
ax.text(split_date + pd.Timedelta(days=15), ax.get_ylim()[1] * 0.92, "Test", ha="left", fontsize=10, fontstyle="italic")

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Daily Overload Rate (%)", fontsize=11)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_title("Daily Overload Rate Across All Transformers (7-day rolling average)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig8_overload_rate.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig8_overload_rate.png")



## 7. Overload events by season

Share of overload events falling in each season.

In [0]:
# =============================================================================
# CELL 6 — Figure 9: Seasonal Pie Chart (Overload Events)
# =============================================================================

df_season = df.withColumn("month", F.month(TS_COL)).withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

season_stats = (
    df_season.withColumn(
        "is_overload",
        F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)
    )
    .groupBy("season")
    .agg(F.sum("is_overload").alias("overload_count"))
    .toPandas()
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_stats["season"] = pd.Categorical(season_stats["season"], categories=season_order, ordered=True)
season_stats = season_stats.sort_values("season")

fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
colors = ["#4A90D9", "#7CB342", "#FFA726", "#8D6E63"]

wedges, texts, autotexts = ax.pie(
    season_stats["overload_count"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["overload_count"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 11}
)
ax.set_title("Overload Events by Season", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig9_seasonal_pie.png", dpi=300, bbox_inches="tight")
plt.show()

total = season_stats["overload_count"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["overload_count"] / total
    print(f"{row['season']:8s}: {row['overload_count']:,} events ({pct:.1f}%)")
print(f"Saved: /dbfs/tmp/thesis_fig9_seasonal_pie.png")



## 8. SCADA event counts

Daily total event count and the share of measurements that have any event.

In [0]:
# =============================================================================
# CELL 7 — Figure 10: SCADA Event Counts (2-panel)
# =============================================================================
# NOTE: This figure reads from gold_dataset. If events_15m_cnt is not in
# gold_dataset, you may need to read from gold_features (but use raw values,
# not scaled). Adjust SRC as needed.

df_events = spark.read.table(SRC_TABLE)

daily_events = (
    df_events.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.sum("events_15m_cnt").alias("total_events"),
        F.mean(F.when(F.col("events_15m_cnt") > 0, 1).otherwise(0)).alias("pct_with_events"),
        F.count("*").alias("n_rows"),
    )
    .orderBy("date")
    .toPandas()
)
daily_events["date"] = pd.to_datetime(daily_events["date"])
daily_events = daily_events.set_index("date").sort_index()
daily_events["pct_with_events"] *= 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# Panel A — Total event count
ax1 = axes[0]
ax1.bar(daily_events.index, daily_events["total_events"], width=1, color="#2ca02c", alpha=0.7)
ax1.plot(daily_events.index, daily_events["total_events"].rolling(7, center=True).mean(),
         linewidth=2, color="#006400", label="7-day rolling mean")
ax1.set_ylabel("Total Events", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Daily SCADA Event Count Across All Transformers", fontsize=11, fontweight="bold")

# Panel B — Percentage with events
ax2 = axes[1]
ax2.bar(daily_events.index, daily_events["pct_with_events"], width=1, color="#9467bd", alpha=0.7)
ax2.plot(daily_events.index, daily_events["pct_with_events"].rolling(7, center=True).mean(),
         linewidth=2, color="#5B2C6F", label="7-day rolling mean")
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Measurements with Events (%)", fontsize=11)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Percentage of Measurements with Associated Events", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig10_events.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig10_events.png")



## 9. Train/test split — label balance

Positive-label rate for each horizon in the train and test partitions, from `gold_features`.

In [0]:
# =============================================================================
# BONUS — Figure for §3.3.5: Train/Test Split Label Balance
# =============================================================================
# This uses gold_features (has labels)

import numpy as np

gf = spark.table("hive_metastore.gold.gold_features")
split_stats = gf.groupBy("split").agg(
    F.count("*").alias("n_rows"),
    F.sum(F.col("label_4h").cast("int")).alias("pos_4h"),
    F.sum(F.col("label_24h").cast("int")).alias("pos_24h")
).withColumn("rate_4h", F.col("pos_4h") / F.col("n_rows") * 100
).withColumn("rate_24h", F.col("pos_24h") / F.col("n_rows") * 100
).orderBy("split")

stats = {row["split"]: row.asDict() for row in split_stats.collect()}

fig, ax = plt.subplots(figsize=(8, 5))
splits = ["train", "test"]
x = np.arange(len(splits))
width = 0.35

rates_4h = [stats["train"]["rate_4h"], stats["test"]["rate_4h"]]
rates_24h = [stats["train"]["rate_24h"], stats["test"]["rate_24h"]]

bars_4h = ax.bar(x - width/2, rates_4h, width, label="4-hour horizon", color="#2c7fb8")
bars_24h = ax.bar(x + width/2, rates_24h, width, label="24-hour horizon", color="#7fcdbb")

ax.set_ylabel("Positive label rate (%)")
ax.set_xticks(x)
ax.set_xticklabels([
    f"Training\n(Jul 2023 – Mar 2024)\nn = {stats['train']['n_rows']:,}",
    f"Test\n(Apr 2024 – Jul 2024)\nn = {stats['test']['n_rows']:,}"
])
ax.legend(loc="upper right")
ax.set_ylim(0, max(rates_24h) * 1.3)

for bars in [bars_4h, bars_24h]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f}%",
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

fig.text(0.5, 0.02, "Split boundary: 1 April 2024 (24-hour purge gap applied)",
         ha="center", fontsize=9, style="italic")

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig("/dbfs/tmp/thesis_fig_train_test_split.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig_train_test_split.png")